In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
spark = SparkSession.builder.appName('Ingestion').getOrCreate()

In [0]:
schema_yellow_taxi = StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestion_fee', DoubleType(), True)])

In [0]:
schema_green_taxi = StructType([StructField('VendorID', IntegerType(), True), StructField('lpep_pickup_datetime', TimestampType(), True), StructField('lpep_dropoff_datetime', TimestampType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('cbd_congestion_fee', DoubleType(), True)])

In [0]:
schema_fhv_trip = StructType([StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropOff_datetime', TimestampType(), True), StructField('PUlocationID', LongType(), True), StructField('DOlocationID', LongType(), True), StructField('SR_Flag', LongType(), True), StructField('Affiliated_base_number', StringType(), True)])

In [0]:
schema_fhvhv_trip = StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampType(), True), StructField('on_scene_datetime', TimestampType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag', StringType(), True), StructField('shared_match_flag', StringType(), True), StructField('access_a_ride_flag', StringType(), True), StructField('wav_request_flag', StringType(), True), StructField('wav_match_flag', StringType(), True), StructField('cbd_congestion_fee', DoubleType(), True)])

In [0]:
df_yellow_taxi = (spark.readStream
      .format("cloudFiles")
      .schema(schema_yellow_taxi)
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://extlocation@adlsformyproject.dfs.core.windows.net/schema_dir")
      .load('abfss://extlocation@adlsformyproject.dfs.core.windows.net/raw_data/yellow_trip/'))

In [0]:
df_yellow_taxi = df_yellow_taxi.withColumn('ingestion_at', current_timestamp())

In [0]:
(df_yellow_taxi.writeStream.format("delta")
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/nyc_taxi_project/bronze/checkpoint")
 .trigger(availableNow=True)
 .table("nyc_taxi_project.bronze.yellow_taxi"))

In [0]:
df_green_taxi = (spark.readStream
      .format("cloudFiles")
      .schema(schema_green_taxi)
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://extlocation@adlsformyproject.dfs.core.windows.net/schema_dir")
      .load('abfss://extlocation@adlsformyproject.dfs.core.windows.net/raw_data/green_trip/'))

In [0]:
df_green_taxi = df_green_taxi.withColumn('ingestion_at', current_timestamp())

In [0]:
(df_green_taxi.writeStream.format("delta")
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/nyc_taxi_project/bronze/checkpoint")
 .trigger(availableNow=True)
 .table("nyc_taxi_project.bronze.green_taxi"))

In [0]:
df_fhv_trip = (spark.readStream
      .format("cloudFiles")
      .schema(schema_fhv_trip)
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://extlocation@adlsformyproject.dfs.core.windows.net/schema_dir")
      .load('abfss://extlocation@adlsformyproject.dfs.core.windows.net/raw_data/fhv_trip/'))

In [0]:
df_fhv_trip = df_fhv_trip.withColumn('ingestion_at', current_timestamp())

In [0]:
(df_fhv_trip.writeStream.format("delta")
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/nyc_taxi_project/bronze/checkpoint")
 .trigger(availableNow=True)
 .table("nyc_taxi_project.bronze.fhv_trip"))

In [0]:
df_fhvhv_trip = (spark.readStream
      .format("cloudFiles")
      .schema(schema_fhvhv_trip)
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://extlocation@adlsformyproject.dfs.core.windows.net/schema_dir")
      .load('abfss://extlocation@adlsformyproject.dfs.core.windows.net/raw_data/fhvhv_trip/'))

In [0]:
df_fhvhv_trip = df_fhvhv_trip.withColumn('ingestion_at', current_timestamp())

In [0]:
(df_fhvhv_trip.writeStream.format("delta")
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/nyc_taxi_project/bronze/checkpoint")
 .trigger(availableNow=True)
 .table("nyc_taxi_project.bronze.fhvhv_trip"))

In [0]:
%sql
select * from nyc_taxi_project.bronze.fhvhv_trip